# Scrub – NRW Mietdaten

This notebook corresponds to the **Scrub** phase of the OSEMN framework.


In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
while project_root.name != "dis08-miete-nrw" and project_root != project_root.parent:
    project_root = project_root.parent

path_raw = project_root / "data" / "raw" / "immo_data.csv"
path_out = project_root / "data" / "processed" / "nrw_clean.csv"

path_raw.exists(), path_raw


C:\Users\User\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


(True,
 WindowsPath('C:/Users/User/Desktop/DATA INFORMATION SCIENCE/DIS08/dis08-miete-nrw/data/raw/immo_data.csv'))

In [2]:
df = pd.read_csv(path_raw)

df_nrw = df[df["regio1"] == "Nordrhein_Westfalen"].copy()

cols = ["regio2", "baseRent", "totalRent", "livingSpace", "noRooms", "balcony", "hasKitchen", "yearConstructed"]
df_nrw = df_nrw[cols].copy()

df_nrw.shape, df_nrw.isna().sum()


((62863, 8),
 regio2                 0
 baseRent               0
 totalRent          12104
 livingSpace            0
 noRooms                0
 balcony                0
 hasKitchen             0
 yearConstructed    12052
 dtype: int64)

In [3]:
num_cols = ["baseRent", "totalRent", "livingSpace", "noRooms", "yearConstructed"]
for c in num_cols:
    df_nrw[c] = pd.to_numeric(df_nrw[c], errors="coerce")

df_nrw.dtypes


regio2              object
baseRent           float64
totalRent          float64
livingSpace        float64
noRooms            float64
balcony               bool
hasKitchen            bool
yearConstructed    float64
dtype: object

In [4]:
df_nrw_clean = df_nrw[
    (df_nrw["baseRent"] > 0) &
    (df_nrw["livingSpace"] > 0)
].copy()

df_nrw_clean["price_per_sqm"] = df_nrw_clean["baseRent"] / df_nrw_clean["livingSpace"]

df_nrw_clean = df_nrw_clean[
    (df_nrw_clean["price_per_sqm"] > 1) &
    (df_nrw_clean["price_per_sqm"] < 50)
].copy()

df_nrw_clean.shape, df_nrw_clean["price_per_sqm"].describe()


((62822, 9),
 count    62822.000000
 mean         8.046116
 std          3.189897
 min          1.684982
 25%          6.000000
 50%          7.192982
 75%          9.193548
 max         47.647059
 Name: price_per_sqm, dtype: float64)

In [5]:
path_out.parent.mkdir(parents=True, exist_ok=True)
df_nrw_clean.to_csv(path_out, index=False)

path_out.exists(), path_out.resolve()


(True,
 WindowsPath('C:/Users/User/Desktop/DATA INFORMATION SCIENCE/DIS08/dis08-miete-nrw/data/processed/nrw_clean.csv'))